In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# ELEVATER Dataset Downloader (Updated)

This notebook downloads and organizes any dataset from the ELEVATER benchmark.

**Data Source:** https://huggingface.co/datasets/ytaek-oh/elevater

**Paper:** ELEVATER: A Benchmark and Toolkit for Evaluating Language-Augmented Visual Models

**Usage:** Simply specify the dataset name and run all cells!

## 1. Configuration - SPECIFY YOUR DATASET HERE

### Available ELEVATER Datasets:

**Format:** `dataset_name_YYYYMMDD`

Common datasets:
- `rendered_sst2_20210924`
- `kitti_distance_20210924`
- `cifar10_20210924`
- `cifar100_20210924`
- `country211_20210924`
- `mnist_20210924`
- `eurosat_clip_20210924`
- `oxford_flowers102_20210924`
- `stanford_cars_20210924`

Check the full list at: https://huggingface.co/datasets/ytaek-oh/elevater/tree/main

In [ ]:
# ============================================
# CONFIGURE YOUR DATASET HERE
# ============================================

DATASET_NAME = "food_101_20211007"  # Change this to any ELEVATER dataset

# ============================================
# Advanced Options (optional)
# ============================================

BASE_OUTPUT_DIR = "/content/drive/MyDrive/MAVERIC/maveric_cache/food101/datasets/elevater"  # Base directory for organized datasets
DOWNLOAD_DIR = "downloads"         # Temporary download directory
HF_REPO = "ytaek-oh/elevater"      # HuggingFace repository

# Extract clean name (remove date suffix)
CLEAN_NAME = '_'.join(DATASET_NAME.split('_')[:-1]) if '_20' in DATASET_NAME else DATASET_NAME

print(f"Dataset: {DATASET_NAME}")
print(f"Clean name: {CLEAN_NAME}")
print(f"Output directory: {BASE_OUTPUT_DIR}/{CLEAN_NAME}")

Dataset: food_101_20211007
Clean name: food_101
Output directory: /content/drive/MyDrive/MAVERIC/maveric_cache/food101/datasets/elevater/food_101


## 2. Install Dependencies

In [ ]:
!pip install -q huggingface_hub pillow tqdm

## 3. Import Libraries

In [ ]:
import os
import shutil
import json
import zipfile
from pathlib import Path
from tqdm import tqdm
from huggingface_hub import hf_hub_download, list_repo_files
from PIL import Image
import numpy as np
from collections import defaultdict
import matplotlib.pyplot as plt
import random

## 4. Setup Directories

In [ ]:
# Create base directories
BASE_DIR = f"{BASE_OUTPUT_DIR}/{CLEAN_NAME}"
os.makedirs(BASE_DIR, exist_ok=True)
os.makedirs(DOWNLOAD_DIR, exist_ok=True)
os.makedirs(f"{BASE_DIR}/train", exist_ok=True)
os.makedirs(f"{BASE_DIR}/test", exist_ok=True)

print(f"Base directory: {BASE_DIR}")
print(f"Download directory: {DOWNLOAD_DIR}")

Base directory: /content/drive/MyDrive/MAVERIC/maveric_cache/food101/datasets/elevater/food_101
Download directory: downloads


## 5. List Available Files in Repository

In [ ]:
print(f"Fetching file list from {HF_REPO}...\n")

try:
    # List all files in the repository
    all_files = list_repo_files(HF_REPO, repo_type="dataset")

    # Filter files for this dataset
    dataset_files = [f for f in all_files if f.startswith(DATASET_NAME + '/')]

    if not dataset_files:
        print(f"Error: No files found for dataset '{DATASET_NAME}'")
        print(f"\nAvailable datasets in repository:")
        datasets = set()
        for f in all_files:
            if '/' in f:
                datasets.add(f.split('/')[0])
        for ds in sorted(datasets):
            print(f"  - {ds}")
        raise ValueError(f"Dataset '{DATASET_NAME}' not found")

    print(f"Found {len(dataset_files)} files for {DATASET_NAME}:")
    for f in sorted(dataset_files):
        print(f"  - {f}")

except Exception as e:
    print(f"Error: {e}")
    raise

Fetching file list from ytaek-oh/elevater...

Found 5 files for food_101_20211007:
  - food_101_20211007/labels.txt
  - food_101_20211007/test.txt
  - food_101_20211007/train.txt
  - food_101_20211007/train.zip
  - food_101_20211007/val.zip


## 6. Download Dataset Files from HuggingFace

In [ ]:
print("Downloading dataset files...\n")

downloaded_files = {}

for file_path in tqdm(dataset_files, desc="Downloading"):
    try:
        # Download file
        local_path = hf_hub_download(
            repo_id=HF_REPO,
            filename=file_path,
            repo_type="dataset",
            local_dir=DOWNLOAD_DIR
        )

        # Store the filename and its local path
        filename = os.path.basename(file_path)
        downloaded_files[filename] = local_path

    except Exception as e:
        print(f"Error downloading {file_path}: {e}")

print(f"\n✓ Downloaded {len(downloaded_files)} files")
print("\nDownloaded files:")
for filename in sorted(downloaded_files.keys()):
    file_size = os.path.getsize(downloaded_files[filename]) / (1024 * 1024)  # MB
    print(f"  - {filename}: {file_size:.2f} MB")

Downloading:   0%|          | 0/5 [00:00<?, ?it/s]

labels.txt: 0.00B [00:00, ?B/s]

Downloading:  20%|██        | 1/5 [00:00<00:00,  4.50it/s]

test.txt: 0.00B [00:00, ?B/s]

Downloading:  40%|████      | 2/5 [00:00<00:00,  4.80it/s]

...

food_101_20211007/train.zip:   0%|          | 0.00/176M [00:00<?, ?B/s]

Downloading:  80%|████████  | 4/5 [00:05<00:02,  2.24s/it]

food_101_20211007/val.zip:   0%|          | 0.00/1.28G [00:00<?, ?B/s]

Downloading: 100%|██████████| 5/5 [00:16<00:00,  3.25s/it]


✓ Downloaded 5 files

Downloaded files:
  - labels.txt: 0.00 MB
  - test.txt: 0.85 MB
  - train.txt: 2.69 MB
  - train.zip: 168.00 MB
  - val.zip: 1218.73 MB


## 7. Parse Labels File

In [ ]:
def parse_labels_file(labels_path):
    """
    Parse the labels.txt file to get class names
    Format: Usually one class name per line
    """
    class_names = []

    if os.path.exists(labels_path):
        with open(labels_path, 'r', encoding='utf-8') as f:
            for line in f:
                line = line.strip()
                if line:  # Skip empty lines
                    class_names.append(line)

    return class_names

# Find and parse labels file
labels_file = downloaded_files.get('labels.txt')

if labels_file:
    class_names = parse_labels_file(labels_file)
    print(f"Found {len(class_names)} classes in labels.txt:")
    for i, name in enumerate(class_names[:10]):  # Show first 10
        print(f"  {i}: {name}")
    if len(class_names) > 10:
        print(f"  ... and {len(class_names) - 10} more")
else:
    print("Warning: labels.txt not found. Will try to infer classes from data.")
    class_names = None

# Create class directories if we have class names
if class_names:
    for split in ['train', 'test']:
        for class_name in class_names:
            # Clean class name for directory
            clean_name = str(class_name).replace(' ', '_').replace('/', '_').replace('\\', '_')
            os.makedirs(f"{BASE_DIR}/{split}/{clean_name}", exist_ok=True)
    print("\n✓ Class directories created")

Found 101 classes in labels.txt:
  0: apple_pie
  1: baby_back_ribs
  2: baklava
  3: beef_carpaccio
  4: beef_tartare
  5: beet_salad
  6: beignets
  7: bibimbap
  8: bread_pudding
  9: breakfast_burrito
  ... and 91 more

✓ Class directories created


## 8. Parse Split Files and Extract Images

In [ ]:
def parse_split_file(split_txt_path):
    """
    Parse split .txt file to get image filenames and their labels
    Format can vary:
    - "filename.jpg label"
    - "filename.jpg 0" (numeric label)
    - Just "filename.jpg" (labels inferred from zip structure)
    """
    image_labels = []

    if not os.path.exists(split_txt_path):
        return image_labels

    with open(split_txt_path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue

            parts = line.split()

            if len(parts) >= 2:
                # Format: filename label
                filename = parts[0]
                label = parts[1]
                # Try to convert to int if possible
                try:
                    label = int(label)
                except:
                    pass
                image_labels.append((filename, label))
            elif len(parts) == 1:
                # Format: just filename
                filename = parts[0]
                image_labels.append((filename, None))

    return image_labels

def extract_and_organize_split(zip_path, split_txt_path, split_name, class_names):
    """
    Extract images from zip and organize them by class
    """
    print(f"\nProcessing {split_name} split...")

    if not os.path.exists(zip_path):
        print(f"Warning: {zip_path} not found, skipping...")
        return {}

    # Parse the split txt file
    image_labels = parse_split_file(split_txt_path)

    # Create a mapping from filename to label
    filename_to_label = {}
    if image_labels:
        filename_to_label = {filename: label for filename, label in image_labels}
        print(f"Found {len(filename_to_label)} image-label pairs")

    # Extract zip file
    class_counts = defaultdict(int)

    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        file_list = zip_ref.namelist()
        print(f"Extracting {len(file_list)} files...")

        for file_info in tqdm(file_list, desc=f"Organizing {split_name}"):
            # Skip directories
            if file_info.endswith('/'):
                continue

            # Get filename
            filename = os.path.basename(file_info)

            # Skip non-image files
            if not filename.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp', '.gif')):
                continue

            # Determine the label/class
            label = filename_to_label.get(filename)

            # If no label in txt file, try to infer from directory structure in zip
            if label is None and '/' in file_info:
                # Extract class from path (e.g., "class_name/image.jpg")
                parts = file_info.split('/')
                if len(parts) >= 2:
                    label = parts[-2]  # The directory name

            # Get class name
            if label is not None:
                if isinstance(label, int) and class_names and label < len(class_names):
                    class_name = class_names[label]
                else:
                    class_name = str(label)
            else:
                class_name = "unknown"

            # Clean class name
            clean_class_name = str(class_name).replace(' ', '_').replace('/', '_').replace('\\', '_')

            # Create class directory if it doesn't exist
            class_dir = os.path.join(BASE_DIR, split_name, clean_class_name)
            os.makedirs(class_dir, exist_ok=True)

            # Extract and save image
            try:
                # Extract to memory
                img_data = zip_ref.read(file_info)

                # Save to appropriate directory
                output_path = os.path.join(class_dir, filename)

                with open(output_path, 'wb') as out_file:
                    out_file.write(img_data)

                class_counts[clean_class_name] += 1

            except Exception as e:
                print(f"Error processing {file_info}: {e}")

    return dict(class_counts)

# Process each split
stats = {'train': {}, 'test': {}}

# Common split mappings
split_configs = [
    # ('train.zip', 'train.txt', 'train'),
    # ('test.zip', 'test.txt', 'test'),
    # ('train.zip', 'train_ic.txt', 'train'),
    # ('test.zip', 'test_ic.txt', 'test'),
    # ('train_images.zip', 'train_meta.json', 'train'),
    # ('test_images.zip', 'test_meta.json', 'test'),
    # ('img.zip', 'train_meta.json', 'train'),
    # ('img.zip', 'test_meta.json', 'test'),
    # ('valid.zip', 'valid.txt', 'test'),  # Map validation to test
    ('val.zip', 'test.txt', 'test'),
]

for zip_name, txt_name, target_split in split_configs:
    zip_path = downloaded_files.get(zip_name)
    txt_path = downloaded_files.get(txt_name)

    if zip_path:
        class_counts = extract_and_organize_split(zip_path, txt_path, target_split, class_names)

        # Merge counts
        for class_name, count in class_counts.items():
            stats[target_split][class_name] = stats[target_split].get(class_name, 0) + count

print("\n✓ Dataset organization complete!")


Processing test split...
Found 25250 image-label pairs
Extracting 25250 files...


Organizing test: 100%|██████████| 25250/25250 [05:20<00:00, 78.67it/s]


✓ Dataset organization complete!


## 9. Verify Directory Structure

In [ ]:
def print_directory_tree(directory, max_depth=3, current_depth=0, prefix="", max_items=15):
    """Print directory tree structure"""
    if current_depth >= max_depth or not os.path.exists(directory):
        return

    items = sorted(os.listdir(directory))
    dirs = [item for item in items if os.path.isdir(os.path.join(directory, item))]

    # Limit number of directories shown
    show_dirs = dirs[:max_items]
    remaining_dirs = len(dirs) - len(show_dirs)

    for i, dir_name in enumerate(show_dirs):
        is_last = (i == len(show_dirs) - 1) and (remaining_dirs == 0)
        connector = "└── " if is_last else "├── "
        print(f"{prefix}{connector}{dir_name}/")

        # Count files
        dir_path = os.path.join(directory, dir_name)
        try:
            file_count = len([f for f in os.listdir(dir_path) if os.path.isfile(os.path.join(dir_path, f))])
            extension = "    " if is_last else "│   "
            if file_count > 0:
                print(f"{prefix}{extension}({file_count} images)")
        except:
            pass

    if remaining_dirs > 0:
        print(f"{prefix}└── ... and {remaining_dirs} more classes")

print("\n" + "="*60)
print("DIRECTORY STRUCTURE")
print("="*60)
print(f"{BASE_DIR}/")
print_directory_tree(BASE_DIR, max_depth=3)


DIRECTORY STRUCTURE
/content/drive/MyDrive/MAVERIC/maveric_cache/food101/datasets/elevater/food_101/
├── test/
└── train/


## 10. Dataset Statistics

In [ ]:
print("\n" + "="*70)
print(f"DATASET STATISTICS: {CLEAN_NAME.upper()}")
print("="*70)

for split in ['train', 'test']:
    if not stats[split]:
        print(f"\n{split.upper()} SET: No data found")
        continue

    print(f"\n{split.upper()} SET:")
    print("-" * 70)

    total = sum(stats[split].values())
    sorted_classes = sorted(stats[split].items(), key=lambda x: x[1], reverse=True)

    # Show first 20 classes
    for class_name, count in sorted_classes[:20]:
        percentage = (count / total * 100) if total > 0 else 0
        bar_length = int(percentage / 2)
        bar = '█' * bar_length
        print(f"  {class_name:30s}: {count:6d} ({percentage:5.2f}%) {bar}")

    if len(sorted_classes) > 20:
        remaining = sum(count for _, count in sorted_classes[20:])
        print(f"  {'... and others':30s}: {remaining:6d} images in {len(sorted_classes) - 20} classes")

    print(f"\n  {'TOTAL':30s}: {total:6d} images")
    print(f"  {'Number of classes':30s}: {len(stats[split])}")

print("\n" + "="*70)


DATASET STATISTICS: FOOD_101

TRAIN SET: No data found

TEST SET:
----------------------------------------------------------------------
  apple_pie                     :    250 ( 0.99%) 
  baby_back_ribs                :    250 ( 0.99%) 
  baklava                       :    250 ( 0.99%) 
  beef_carpaccio                :    250 ( 0.99%) 
  beef_tartare                  :    250 ( 0.99%) 
  beet_salad                    :    250 ( 0.99%) 
  beignets                      :    250 ( 0.99%) 
  bibimbap                      :    250 ( 0.99%) 
  bread_pudding                 :    250 ( 0.99%) 
  breakfast_burrito             :    250 ( 0.99%) 
  bruschetta                    :    250 ( 0.99%) 
  caesar_salad                  :    250 ( 0.99%) 
  cannoli                       :    250 ( 0.99%) 
  caprese_salad                 :    250 ( 0.99%) 
  carrot_cake                   :    250 ( 0.99%) 
  ceviche                       :    250 ( 0.99%) 
  cheesecake                    :    250 ( 0.9

## 11. Visualize Sample Images

In [ ]:
def visualize_samples(base_dir, split='train', num_classes=5, samples_per_class=4):
    """Visualize sample images from each class"""
    split_dir = os.path.join(base_dir, split)

    if not os.path.exists(split_dir):
        print(f"Split '{split}' not found")
        return

    classes = sorted([d for d in os.listdir(split_dir) if os.path.isdir(os.path.join(split_dir, d))])

    if len(classes) > num_classes:
        classes = random.sample(classes, num_classes)

    for class_name in classes:
        class_dir = os.path.join(split_dir, class_name)
        images = [f for f in os.listdir(class_dir) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]

        if len(images) == 0:
            continue

        num_samples = min(samples_per_class, len(images))
        samples = random.sample(images, num_samples)

        fig, axes = plt.subplots(1, num_samples, figsize=(15, 3))
        fig.suptitle(f'{split.upper()} - Class: {class_name}', fontsize=14, fontweight='bold')

        if num_samples == 1:
            axes = [axes]

        for idx, img_name in enumerate(samples):
            img_path = os.path.join(class_dir, img_name)
            try:
                img = Image.open(img_path)
                axes[idx].imshow(img)
                axes[idx].axis('off')
                axes[idx].set_title(img_name[:20], fontsize=8)
            except Exception as e:
                print(f"Error loading {img_path}: {e}")

        plt.tight_layout()
        plt.show()

print("Sample Images:")
print("="*60)
visualize_samples(BASE_DIR, split='train', num_classes=5, samples_per_class=4)
visualize_samples(BASE_DIR, split='test', num_classes=3, samples_per_class=3)

## 12. Class Distribution Visualization

In [ ]:
def plot_class_distribution(stats, max_classes=20):
    """Plot class distribution"""
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))

    for idx, split in enumerate(['train', 'test']):
        if not stats[split]:
            axes[idx].text(0.5, 0.5, f'No {split} data', ha='center', va='center', fontsize=14)
            axes[idx].set_title(f'{split.upper()} Set', fontsize=14, fontweight='bold')
            continue

        sorted_items = sorted(stats[split].items(), key=lambda x: x[1], reverse=True)

        if len(sorted_items) > max_classes:
            sorted_items = sorted_items[:max_classes]
            other_count = sum(count for _, count in sorted(stats[split].items(), key=lambda x: x[1], reverse=True)[max_classes:])
            if other_count > 0:
                sorted_items.append(('Others', other_count))

        classes = [item[0] for item in sorted_items]
        counts = [item[1] for item in sorted_items]

        colors = plt.cm.Set3(np.linspace(0, 1, len(classes)))
        bars = axes[idx].bar(range(len(classes)), counts, color=colors)
        axes[idx].set_title(f'{split.upper()} Set Distribution', fontsize=14, fontweight='bold')
        axes[idx].set_xlabel('Class', fontsize=12)
        axes[idx].set_ylabel('Number of Images', fontsize=12)
        axes[idx].set_xticks(range(len(classes)))
        axes[idx].set_xticklabels([c[:15] for c in classes], rotation=45, ha='right', fontsize=9)
        axes[idx].grid(axis='y', alpha=0.3)

        for bar, count in zip(bars, counts):
            height = bar.get_height()
            axes[idx].text(bar.get_x() + bar.get_width()/2., height,
                          f'{count}', ha='center', va='bottom', fontsize=8, fontweight='bold')

    plt.tight_layout()
    plt.show()

if any(stats['train'].values()) or any(stats['test'].values()):
    print("\nClass Distribution:")
    plot_class_distribution(stats, max_classes=20)

## 13. Save Dataset Metadata

In [ ]:
# Get all unique classes
all_classes = sorted(set(list(stats['train'].keys()) + list(stats['test'].keys())))

dataset_info = {
    'dataset_name': CLEAN_NAME,
    'full_name': DATASET_NAME,
    'source': f'ELEVATER Benchmark - https://huggingface.co/datasets/{HF_REPO}',
    'structure': f'data/elevater/{CLEAN_NAME}/train|test/class/',
    'classes': class_names if class_names else all_classes,
    'num_classes': len(class_names) if class_names else len(all_classes),
    'statistics': stats,
    'total_train_images': sum(stats['train'].values()),
    'total_test_images': sum(stats['test'].values()),
}

info_file = os.path.join(BASE_DIR, 'dataset_info.json')
with open(info_file, 'w') as f:
    json.dump(dataset_info, f, indent=2)

print("\n" + "="*70)
print("DATASET ORGANIZATION COMPLETE!")
print("="*70)
print(f"\n✓ Dataset: {CLEAN_NAME}")
print(f"✓ Full name: {DATASET_NAME}")
print(f"✓ Location: {BASE_DIR}")
print(f"✓ Classes: {len(all_classes)}")
print(f"✓ Train images: {sum(stats['train'].values())}")
print(f"✓ Test images: {sum(stats['test'].values())}")
print(f"✓ Metadata: {info_file}")
print("\nDataset ready for ELEVATER benchmark evaluation!")


DATASET ORGANIZATION COMPLETE!

✓ Dataset: rendered_sst2
✓ Full name: rendered_sst2_20210924
✓ Location: /content/drive/MyDrive/MAVERIC/maveric_cache/rendered_sst2/datasets/elevater/rendered_sst2
✓ Classes: 2
✓ Train images: 6920
✓ Test images: 1821
✓ Metadata: /content/drive/MyDrive/MAVERIC/maveric_cache/rendered_sst2/datasets/elevater/rendered_sst2/dataset_info.json

Dataset ready for ELEVATER benchmark evaluation!


## 14. Optional: Compress and Download

In [ ]:
# Uncomment to compress and download
# import shutil
# archive_name = f"{CLEAN_NAME}_organized"
# print(f"Creating archive: {archive_name}.zip")
# shutil.make_archive(archive_name, 'zip', BASE_DIR)
# print(f"✓ Archive created")
#
# from google.colab import files
# files.download(f"{archive_name}.zip")

In [ ]:
import sys
from pathlib import Path


# Caltech101 class names in order (000-100)
CLASS_NAMES = [
    'accordion', 'airplanes', 'anchor', 'ant', 'background_google', 'barrel', 'bass', 'beaver',
    'binocular', 'bonsai', 'brain', 'brontosaurus', 'buddha', 'butterfly',
    'camera', 'cannon', 'car_side', 'ceiling_fan', 'cellphone', 'chair',
    'chandelier', 'cougar_body', 'cougar_face', 'crab', 'crayfish', 'crocodile',
    'crocodile_head', 'cup', 'dalmatian', 'dollar_bill', 'dolphin', 'dragonfly',
    'electric_guitar', 'elephant', 'emu', 'euphonium', 'ewer', 'faces_easy',
    'faces', 'ferry', 'flamingo', 'flamingo_head', 'garfield', 'gerenuk',
    'gramophone', 'grand_piano', 'hawksbill', 'headphone', 'hedgehog',
    'helicopter', 'ibis', 'inline_skate', 'joshua_tree', 'kangaroo', 'ketch',
    'lamp', 'laptop', 'leopards', 'llama', 'lobster', 'lotus', 'mandolin',
    'mayfly', 'menorah', 'metronome', 'minaret', 'motorbikes', 'nautilus',
    'octopus', 'okapi', 'pagoda', 'panda', 'pigeon', 'pizza', 'platypus',
    'pyramid', 'revolver', 'rhino', 'rooster', 'saxophone', 'schooner',
    'scissors', 'scorpion', 'sea_horse', 'snoopy', 'soccer_ball', 'stapler',
    'starfish', 'stegosaurus', 'stop_sign', 'strawberry', 'sunflower', 'tick',
    'trilobite', 'umbrella', 'watch', 'water_lilly', 'wheelchair', 'wild_cat',
    'windsor_chair', 'wrench', 'yin_yang'
]


def rename_folders(dataset_path):
    """Rename folders from numeric to class names."""
    dataset_path = Path(dataset_path)

    if not dataset_path.exists():
        print(f"Error: Path not found: {dataset_path}")
        return

    renamed = 0
    for i, class_name in enumerate(CLASS_NAMES):
        # Try zero-padded format (000, 001, etc.)
        numeric_folder = dataset_path / str(i).zfill(3)

        # If not found, try without padding (0, 1, etc.)
        if not numeric_folder.exists():
            numeric_folder = dataset_path / str(i)

        # Skip if numeric folder doesn't exist
        if not numeric_folder.exists():
            continue

        # Rename to class name
        target_folder = dataset_path / class_name

        # Skip if target already exists
        if target_folder.exists():
            print(f"Skip: {class_name} (already exists)")
            continue

        numeric_folder.rename(target_folder)
        renamed += 1
        print(f"✓ {numeric_folder.name} → {class_name}")

    print(f"\nDone! Renamed {renamed} folders.")


rename_folders("/content/drive/MyDrive/MAVERIC/maveric_cache/caltech_101/datasets/elevater/caltech_101/test")

✓ 000 → accordion
✓ 001 → airplanes
✓ 002 → anchor
✓ 003 → ant
✓ 004 → background_google
✓ 005 → barrel
✓ 006 → bass
✓ 007 → beaver
✓ 008 → binocular
✓ 009 → bonsai
✓ 010 → brain
✓ 011 → brontosaurus
✓ 012 → buddha
✓ 013 → butterfly
✓ 014 → camera
✓ 015 → cannon
✓ 016 → car_side
✓ 017 → ceiling_fan
✓ 018 → cellphone
✓ 019 → chair
✓ 020 → chandelier
✓ 021 → cougar_body
✓ 022 → cougar_face
✓ 023 → crab
✓ 024 → crayfish
✓ 025 → crocodile
✓ 026 → crocodile_head
✓ 027 → cup
✓ 028 → dalmatian
✓ 029 → dollar_bill
✓ 030 → dolphin
✓ 031 → dragonfly
✓ 032 → electric_guitar
✓ 033 → elephant
✓ 034 → emu
✓ 035 → euphonium
✓ 036 → ewer
✓ 037 → faces_easy
✓ 038 → faces
✓ 039 → ferry
✓ 040 → flamingo
✓ 041 → flamingo_head
✓ 042 → garfield
✓ 043 → gerenuk
✓ 044 → gramophone
✓ 045 → grand_piano
✓ 046 → hawksbill
✓ 047 → headphone
✓ 048 → hedgehog
✓ 049 → helicopter
✓ 050 → ibis
✓ 051 → inline_skate
✓ 052 → joshua_tree
✓ 053 → kangaroo
✓ 054 → ketch
✓ 055 → lamp
✓ 056 → laptop
✓ 057 → leopards
✓ 058 → ll

In [ ]:
import sys
from pathlib import Path


# Target class names (Country211 example - replace with your dataset's class names)
CLASS_NAMES = [
    'Andorra',
    'United Arab Emirates',
    'Afghanistan',
    'Antigua and Barbuda',
    'Anguilla',
    'Albania',
    'Armenia',
    'Angola',
    'Antarctica',
    'Argentina',
    'Austria',
    'Australia',
    'Aruba',
    'Aland Islands',
    'Azerbaijan',
    'Bosnia and Herzegovina',
    'Barbados',
    'Bangladesh',
    'Belgium',
    'Burkina Faso',
    'Bulgaria',
    'Bahrain',
    'Benin',
    'Bermuda',
    'Brunei Darussalam',
    'Bolivia',
    'Bonaire, Saint Eustatius and Saba',
    'Brazil',
    'Bahamas',
    'Bhutan',
    'Botswana',
    'Belarus',
    'Belize',
    'Canada',
    'DR Congo',
    'Central African Republic',
    'Switzerland',
    "Cote d'Ivoire",
    'Cook Islands',
    'Chile',
    'Cameroon',
    'China',
    'Colombia',
    'Costa Rica',
    'Cuba',
    'Cabo Verde',
    'Curacao',
    'Cyprus',
    'Czech Republic',
    'Germany',
    'Denmark',
    'Dominica',
    'Dominican Republic',
    'Algeria',
    'Ecuador',
    'Estonia',
    'Egypt',
    'Spain',
    'Ethiopia',
    'Finland',
    'Fiji',
    'Falkland Islands',
    'Faeroe Islands',
    'France',
    'Gabon',
    'United Kingdom',
    'Grenada',
    'Georgia',
    'French Guiana',
    'Guernsey',
    'Ghana',
    'Gibraltar',
    'Greenland',
    'Gambia',
    'Guadeloupe',
    'Greece',
    'South Georgia and South Sandwich Is.',
    'Guatemala',
    'Guam',
    'Guyana',
    'Hong Kong',
    'Honduras',
    'Croatia',
    'Haiti',
    'Hungary',
    'Indonesia',
    'Ireland',
    'Israel',
    'Isle of Man',
    'India',
    'Iraq',
    'Iran',
    'Iceland',
    'Italy',
    'Jersey',
    'Jamaica',
    'Jordan',
    'Japan',
    'Kenya',
    'Kyrgyz Republic',
    'Cambodia',
    'St. Kitts and Nevis',
    'North Korea',
    'South Korea',
    'Kuwait',
    'Cayman Islands',
    'Kazakhstan',
    'Laos',
    'Lebanon',
    'St. Lucia',
    'Liechtenstein',
    'Sri Lanka',
    'Liberia',
    'Lithuania',
    'Luxembourg',
    'Latvia',
    'Libya',
    'Morocco',
    'Monaco',
    'Moldova',
    'Montenegro',
    'Saint-Martin',
    'Madagascar',
    'Macedonia',
    'Mali',
    'Myanmar',
    'Mongolia',
    'Macau',
    'Martinique',
    'Mauritania',
    'Malta',
    'Mauritius',
    'Maldives',
    'Malawi',
    'Mexico',
    'Malaysia',
    'Mozambique',
    'Namibia',
    'New Caledonia',
    'Nigeria',
    'Nicaragua',
    'Netherlands',
    'Norway',
    'Nepal',
    'New Zealand',
    'Oman',
    'Panama',
    'Peru',
    'French Polynesia',
    'Papua New Guinea',
    'Philippines',
    'Pakistan',
    'Poland',
    'Puerto Rico',
    'Palestine',
    'Portugal',
    'Palau',
    'Paraguay',
    'Qatar',
    'Reunion',
    'Romania',
    'Serbia',
    'Russia',
    'Rwanda',
    'Saudi Arabia',
    'Solomon Islands',
    'Seychelles',
    'Sudan',
    'Sweden',
    'Singapore',
    'St. Helena',
    'Slovenia',
    'Svalbard and Jan Mayen Islands',
    'Slovakia',
    'Sierra Leone',
    'San Marino',
    'Senegal',
    'Somalia',
    'South Sudan',
    'El Salvador',
    'Sint Maarten',
    'Syria',
    'Eswatini',
    'Togo',
    'Thailand',
    'Tajikistan',
    'Timor-Leste',
    'Turkmenistan',
    'Tunisia',
    'Tonga',
    'Turkey',
    'Trinidad and Tobago',
    'Taiwan',
    'Tanzania',
    'Ukraine',
    'Uganda',
    'United States',
    'Uruguay',
    'Uzbekistan',
    'Vatican',
    'Venezuela',
    'British Virgin Islands',
    'United States Virgin Islands',
    'Vietnam',
    'Vanuatu',
    'Samoa',
    'Kosovo',
    'Yemen',
    'South Africa',
    'Zambia',
    'Zimbabwe'
]


def read_labels(labels_file):
    """Read current folder names from labels file."""
    labels_path = Path(labels_file)

    if not labels_path.exists():
        print(f"Error: Labels file not found: {labels_file}")
        sys.exit(1)

    with open(labels_path, 'r') as f:
        labels = [line.strip() for line in f if line.strip()]

    return labels


def rename_folders(dataset_path, labels_file):
    """Rename folders based on labels file mapping."""
    dataset_path = Path(dataset_path)

    if not dataset_path.exists():
        print(f"Error: Path not found: {dataset_path}")
        return

    # Read current folder names from labels file
    current_labels = read_labels(labels_file)

    # Check if counts match
    if len(current_labels) != len(CLASS_NAMES):
        print(f"Warning: Label count mismatch!")
        print(f"  Labels file: {len(current_labels)} items")
        print(f"  Target names: {len(CLASS_NAMES)} items")
        print("  Proceeding with available mappings...")

    # Rename folders
    renamed = 0
    max_items = min(len(current_labels), len(CLASS_NAMES))

    for i in range(max_items):
        current_name = current_labels[i]
        target_name = CLASS_NAMES[i]

        current_folder = dataset_path / current_name

        # Skip if current folder doesn't exist
        if not current_folder.exists():
            print(f"Skip: {current_name} (not found)")
            continue

        # Skip if target already exists
        target_folder = dataset_path / target_name
        if target_folder.exists():
            print(f"Skip: {current_name} → {target_name} (target exists)")
            continue

        # Rename
        current_folder.rename(target_folder)
        renamed += 1
        print(f"✓ {current_name} → {target_name}")

    print(f"\nDone! Renamed {renamed} folders.")


rename_folders("/content/drive/MyDrive/MAVERIC/maveric_cache/country211/datasets/elevater/country211/test",
               "/content/drive/MyDrive/MAVERIC/maveric_cache/country211/datasets/elevater/country211/labels.txt")



✓ AD → Andorra
✓ AE → United Arab Emirates
✓ AF → Afghanistan
✓ AG → Antigua and Barbuda
✓ AI → Anguilla
✓ AL → Albania
✓ AM → Armenia
✓ AO → Angola
✓ AQ → Antarctica
✓ AR → Argentina
✓ AT → Austria
✓ AU → Australia
✓ AW → Aruba
✓ AX → Aland Islands
✓ AZ → Azerbaijan
✓ BA → Bosnia and Herzegovina
✓ BB → Barbados
✓ BD → Bangladesh
✓ BE → Belgium
✓ BF → Burkina Faso
✓ BG → Bulgaria
✓ BH → Bahrain
✓ BJ → Benin
✓ BM → Bermuda
✓ BN → Brunei Darussalam
✓ BO → Bolivia
✓ BQ → Bonaire, Saint Eustatius and Saba
✓ BR → Brazil
✓ BS → Bahamas
✓ BT → Bhutan
✓ BW → Botswana
✓ BY → Belarus
✓ BZ → Belize
✓ CA → Canada
✓ CD → DR Congo
✓ CF → Central African Republic
✓ CH → Switzerland
✓ CI → Cote d'Ivoire
✓ CK → Cook Islands
✓ CL → Chile
✓ CM → Cameroon
✓ CN → China
✓ CO → Colombia
✓ CR → Costa Rica
✓ CU → Cuba
✓ CV → Cabo Verde
✓ CW → Curacao
✓ CY → Cyprus
✓ CZ → Czech Republic
✓ DE → Germany
✓ DK → Denmark
✓ DM → Dominica
✓ DO → Dominican Republic
✓ DZ → Algeria
✓ EC → Ecuador
✓ EE → Estonia
✓ EG → Eg

In [ ]:
"""
Download and organize VOC2007 dataset from HuggingFace (ZIP + TXT format).
URL: https://huggingface.co/datasets/ytaek-oh/elevater/tree/main/voc2007_20211007

Files structure:
- test.zip (445 MB)
- train.zip (454 MB)
- test_ic.txt (276 kB) - image paths and labels
- train_ic.txt (137 kB)
- val_ic.txt (137 kB)
- labels.txt (153 Bytes) - class names

Usage: python download_voc2007_updated.py [--output_dir ./data]
"""

import os
import argparse
import zipfile
from pathlib import Path
from huggingface_hub import hf_hub_download
import shutil


# VOC2007 class names (20 classes)
VOC2007_CLASSES = [
    'aeroplane', 'bicycle', 'bird', 'boat', 'bottle', 'bus', 'car', 'cat',
    'chair', 'cow', 'diningtable', 'dog', 'horse', 'motorbike', 'person',
    'pottedplant', 'sheep', 'sofa', 'train', 'tvmonitor'
]


def download_file(repo_id, subfolder, filename, output_dir):
    """Download a single file from HuggingFace."""
    print(f"  Downloading: {filename}")

    file_path = f"{subfolder}/{filename}"

    local_path = hf_hub_download(
        repo_id=repo_id,
        repo_type="dataset",
        filename=file_path,
        local_dir=output_dir,
        local_dir_use_symlinks=False
    )

    return Path(local_path)


def download_voc2007_files(repo_id="ytaek-oh/elevater",
                           subfolder="voc2007_20211007",
                           output_dir="./data"):
    """Download VOC2007 files from HuggingFace."""
    print("="*70)
    print("Downloading VOC2007 from HuggingFace")
    print("="*70)
    print(f"Repository: {repo_id}")
    print(f"Subfolder: {subfolder}")
    print()

    output_path = Path(output_dir) / "voc2007_temp"
    output_path.mkdir(parents=True, exist_ok=True)

    # Files to download
    files_to_download = [
        'test.zip',
        'train.zip',
        'test_ic.txt',
        'train_ic.txt',
        'labels.txt'
    ]

    downloaded_files = {}

    for filename in files_to_download:
        try:
            local_path = download_file(repo_id, subfolder, filename, output_path)
            downloaded_files[filename] = local_path
            print(f"    ✓ Downloaded: {filename}")
        except Exception as e:
            print(f"    ⚠️  Failed to download {filename}: {e}")

    print(f"\n✅ Downloaded {len(downloaded_files)} files")
    return output_path / subfolder, downloaded_files


def extract_zip_files(downloaded_files, base_path):
    """Extract ZIP files."""
    print("\n" + "="*70)
    print("Extracting ZIP files")
    print("="*70)

    extracted_dirs = {}

    for filename, filepath in downloaded_files.items():
        if filename.endswith('.zip'):
            print(f"\nExtracting: {filename}")

            # Create extraction directory
            extract_dir = filepath.parent / filename.replace('.zip', '')
            extract_dir.mkdir(exist_ok=True)

            try:
                with zipfile.ZipFile(filepath, 'r') as zip_ref:
                    zip_ref.extractall(extract_dir)

                # Count extracted files
                extracted_count = len(list(extract_dir.rglob('*')))
                print(f"  ✓ Extracted {extracted_count} items to: {extract_dir.name}/")

                extracted_dirs[filename.replace('.zip', '')] = extract_dir

            except Exception as e:
                print(f"  ❌ Error extracting {filename}: {e}")

    return extracted_dirs


def read_label_file(label_file_path):
    """Read image-label mappings from _ic.txt file."""
    mappings = []

    with open(label_file_path, 'r') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue

            # Format is typically: image_path label
            # or: image_path label1 label2 ... (multi-label)
            parts = line.split()

            if len(parts) >= 2:
                image_path = parts[0]
                # VOC2007 can have multiple labels, take the first or all
                labels = [int(l) for l in parts[1:] if l.isdigit() or (l.startswith('-') and l[1:].isdigit())]

                # For multi-label, we'll assign to first positive label
                # Negative labels mean the object is absent
                positive_labels = [l for l in labels if l >= 0]

                if positive_labels:
                    mappings.append((image_path, positive_labels[0]))

    return mappings


def organize_voc2007(extracted_dirs, downloaded_files, output_dir="./data"):
    """Organize VOC2007 dataset into class folders."""
    print("\n" + "="*70)
    print("Organizing VOC2007 dataset for MAVERIC")
    print("="*70)

    target_dir = Path(output_dir) / "voc2007"

    # Create class directories
    print("\n📁 Creating directory structure...")
    for split in ['test', 'train']:
        for class_name in VOC2007_CLASSES:
            (target_dir / split / class_name).mkdir(parents=True, exist_ok=True)

    print(f"  ✓ Created {len(VOC2007_CLASSES)} class directories for each split")

    # Process each split
    stats = {'test': 0, 'train': 0}

    for split in ['test', 'train']:
        label_file = downloaded_files.get(f'{split}_ic.txt')
        images_dir = extracted_dirs.get(split)

        if not label_file or not images_dir:
            print(f"\n⚠️  Skipping {split} split - missing files")
            continue

        print(f"\n📋 Processing {split} split...")
        print(f"  Label file: {label_file.name}")
        print(f"  Images dir: {images_dir}")

        # Read label mappings
        mappings = read_label_file(label_file)
        print(f"  Found {len(mappings)} image-label pairs")

        # Copy images to class folders
        copied = 0
        skipped = 0

        for image_path, label_idx in mappings:
            # Get class name
            if label_idx < 0 or label_idx >= len(VOC2007_CLASSES):
                skipped += 1
                continue

            class_name = VOC2007_CLASSES[label_idx]

            # Find source image
            # The path might be relative or just filename
            source_img = images_dir / image_path
            if not source_img.exists():
                # Try just the filename
                source_img = images_dir / Path(image_path).name

            # Try searching in subdirectories
            if not source_img.exists():
                found_files = list(images_dir.rglob(Path(image_path).name))
                if found_files:
                    source_img = found_files[0]

            if not source_img.exists():
                skipped += 1
                continue

            # Copy to target class directory
            target_img = target_dir / split / class_name / source_img.name

            try:
                shutil.copy2(source_img, target_img)
                copied += 1

                if copied % 100 == 0:
                    print(f"    Progress: {copied} images copied...")

            except Exception as e:
                skipped += 1

        print(f"  ✓ Copied {copied} images")
        if skipped > 0:
            print(f"  ⚠️  Skipped {skipped} images")

        stats[split] = copied

    # Print summary
    print_summary(target_dir, stats)

    return target_dir


def print_summary(target_dir, stats):
    """Print dataset summary."""
    print("\n" + "="*70)
    print("Dataset Summary")
    print("="*70)

    for split in ['test', 'train']:
        split_dir = target_dir / split
        print(f"\n{split.capitalize()} Split:")

        class_counts = {}
        total = 0

        for class_name in VOC2007_CLASSES:
            class_dir = split_dir / class_name
            if class_dir.exists():
                n_images = len(list(class_dir.glob('*.jpg')) +
                             list(class_dir.glob('*.jpeg')) +
                             list(class_dir.glob('*.png')) +
                             list(class_dir.glob('*.JPEG')))

                if n_images > 0:
                    class_counts[class_name] = n_images
                    total += n_images

        # Print top 10 classes
        sorted_classes = sorted(class_counts.items(), key=lambda x: x[1], reverse=True)
        for class_name, count in sorted_classes[:10]:
            print(f"  {class_name:15s}: {count:4d} images")

        if len(sorted_classes) > 10:
            print(f"  ... and {len(sorted_classes) - 10} more classes")

        print(f"  {'Total':15s}: {total:4d} images")
        print(f"  {'Classes with data':15s}: {len(class_counts)}/{len(VOC2007_CLASSES)}")


def cleanup_temp_files(temp_dir):
    """Clean up temporary download directory."""
    print("\n🧹 Cleaning up temporary files...")

    try:
        if temp_dir.exists():
            shutil.rmtree(temp_dir)
            print(f"  ✓ Removed: {temp_dir}")
    except Exception as e:
        print(f"  ⚠️  Could not remove temp dir: {e}")


# Download files
temp_path, downloaded_files = download_voc2007_files(output_dir="/content/drive/MyDrive/MAVERIC/maveric_cache/voc2007/datasets/elevater")

# Extract ZIP files
extracted_dirs = extract_zip_files(downloaded_files, temp_path)

# Organize dataset
target_dir = organize_voc2007(extracted_dirs, downloaded_files, "/content/drive/MyDrive/MAVERIC/maveric_cache/voc2007/datasets/elevater")


Repository: ytaek-oh/elevater
Subfolder: voc2007_20211007

  Downloading: test.zip


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:202: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(


    ✓ Downloaded: test.zip
  Downloading: train.zip
    ✓ Downloaded: train.zip
  Downloading: test_ic.txt
    ✓ Downloaded: test_ic.txt
  Downloading: train_ic.txt
    ✓ Downloaded: train_ic.txt
  Downloading: labels.txt
    ✓ Downloaded: labels.txt

✅ Downloaded 5 files

Extracting ZIP files

Extracting: test.zip


In [ ]:
#!/usr/bin/env python3
"""
Organize already extracted VOC2007 files for MAVERIC.
Use this if you've already downloaded and extracted the ZIP files.

Usage: python organize_extracted_voc2007.py <extracted_folder> [--output_dir ./data]

Example:
  python organize_extracted_voc2007.py /content/drive/MyDrive/MAVERIC/maveric_cache/voc2007/datasets/elevater/voc2007_temp/voc2007_20211007/
"""

import argparse
import shutil
from pathlib import Path


VOC2007_CLASSES = [
    'aeroplane', 'bicycle', 'bird', 'boat', 'bottle', 'bus', 'car', 'cat',
    'chair', 'cow', 'diningtable', 'dog', 'horse', 'motorbike', 'person',
    'pottedplant', 'sheep', 'sofa', 'train', 'tvmonitor'
]


def find_jpeg_images_dir(base_dir):
    """Find the JPEGImages directory."""
    base_path = Path(base_dir)

    # Search for JPEGImages directory
    jpeg_dirs = list(base_path.rglob("JPEGImages"))

    if jpeg_dirs:
        print(f"  Found JPEGImages: {jpeg_dirs[0]}")
        return jpeg_dirs[0]

    # Look for directories with many images
    for root, dirs, files in base_path.walk():
        img_files = [f for f in files if f.lower().endswith(('.jpg', '.jpeg'))]
        if len(img_files) > 100:
            print(f"  Found image directory: {root}")
            return Path(root)

    return None


def read_label_file(label_file):
    mappings = []

    with open(label_file, 'r') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue

            # Split path and labels (split on first space only)
            parts = line.rsplit(' ', 1)
            if len(parts) != 2:
                continue

            image_path, labels_str = parts

            # Remove "test.zip@" prefix
            if '@' in image_path:
                image_path = image_path.split('@', 1)[1]

            # Extract filename
            filename = Path(image_path).name

            # Parse comma-separated labels
            labels = [int(l.strip()) for l in labels_str.split(',')]

            if labels:
                mappings.append((filename, labels[0]))  # Use first label

    return mappings


def organize_split(base_dir, split_name, output_dir):
    """Organize one split."""
    base_path = Path(base_dir)
    target_dir = Path(output_dir) / "voc2007" / split_name

    print(f"\n{'='*70}")
    print(f"Processing {split_name.upper()} split")
    print('='*70)

    # Create class directories
    for class_name in VOC2007_CLASSES:
        (target_dir / class_name).mkdir(parents=True, exist_ok=True)

    # Find label file
    label_files = list(base_path.glob(f"{split_name}_ic.txt"))
    if not label_files:
        label_files = list(base_path.glob(f"**/{split_name}_ic.txt"))

    if not label_files:
        print(f"❌ Label file not found: {split_name}_ic.txt")
        return 0

    label_file = label_files[0]
    print(f"Label file: {label_file}")

    # Find images directory
    split_dir = base_path / split_name
    if not split_dir.exists():
        # Try to find it
        split_dirs = list(base_path.glob(f"**/{split_name}"))
        if split_dirs:
            split_dir = split_dirs[0]
        else:
            print(f"❌ Split directory not found: {split_name}")
            return 0

    print(f"Split directory: {split_dir}")

    jpeg_dir = find_jpeg_images_dir(split_dir)

    if not jpeg_dir:
        print(f"❌ Could not find JPEGImages directory")
        return 0

    # Read labels
    mappings = read_label_file(label_file)
    print(f"Found {len(mappings)} image-label pairs")

    # Copy images
    copied = 0
    skipped = 0

    print(f"\nCopying images to class folders...")

    for filename, label_idx in mappings:
        if label_idx >= len(VOC2007_CLASSES):
            skipped += 1
            continue

        class_name = VOC2007_CLASSES[label_idx]
        source_img = jpeg_dir / filename

        if not source_img.exists():
            skipped += 1
            continue

        target_img = target_dir / class_name / filename

        try:
            shutil.copy2(source_img, target_img)
            copied += 1

            if copied % 500 == 0:
                print(f"  Progress: {copied}/{len(mappings)} images")

        except Exception as e:
            print(f"  ⚠️  Error copying {filename}: {e}")
            skipped += 1

    print(f"\n✓ Copied {copied} images")
    if skipped > 0:
        print(f"⚠️  Skipped {skipped} images")

    return copied


def print_summary(output_dir):
    """Print summary."""
    print(f"\n{'='*70}")
    print("Dataset Summary")
    print('='*70)

    voc_dir = Path(output_dir) / "voc2007"

    for split in ['test', 'train']:
        split_dir = voc_dir / split

        if not split_dir.exists():
            continue

        print(f"\n{split.capitalize()} Split:")

        total = 0
        class_counts = []

        for class_name in VOC2007_CLASSES:
            class_dir = split_dir / class_name
            if class_dir.exists():
                n_images = len(list(class_dir.glob('*.jpg')) +
                             list(class_dir.glob('*.jpeg')))
                if n_images > 0:
                    class_counts.append((class_name, n_images))
                    total += n_images

        # Show top 10
        class_counts.sort(key=lambda x: x[1], reverse=True)
        for class_name, count in class_counts[:10]:
            print(f"  {class_name:15s}: {count:4d} images")

        if len(class_counts) > 10:
            print(f"  ... and {len(class_counts) - 10} more")

        print(f"  {'Total':15s}: {total:4d} images")


# args = parser.parse_args()

source_path = Path("/content/drive/MyDrive/MAVERIC/maveric_cache/voc2007/datasets/elevater/voc2007_temp/voc2007_20211007/")

# if not source_path.exists():
#     print(f"❌ Source directory not found: {args.source_dir}")
#     return

# print("="*70)
# print("VOC2007 Organization Script (For Extracted Files)")
# print("="*70)
# print(f"Source: {args.source_dir}")
# print(f"Output: {args.output_dir}/voc2007/")
# print("="*70)

# Organize each split
total = 0
for split in ['test', 'train']:
    copied = organize_split("/content/drive/MyDrive/MAVERIC/maveric_cache/voc2007/datasets/elevater/voc2007_temp/voc2007_20211007/", split, "/content/drive/MyDrive/MAVERIC/maveric_cache/voc2007/datasets/elevater/voc2007/")
    total += copied

# Summary
print_summary("/content/drive/MyDrive/MAVERIC/maveric_cache/voc2007/datasets/elevater/voc2007/")

print(f"\n{'='*70}")
print(f"✅ Done! Organized {total} images")
print('='*70)
print(f"\n📂 Dataset: {"/content/drive/MyDrive/MAVERIC/maveric_cache/voc2007/datasets/elevater/voc2007/"}/voc2007/")
print("\nUse with MAVERIC:")
print("  from maveric.datasets.elevater_datasets import ELEVATERDataset")
print("  dataset = ELEVATERDataset('voc2007', root='./data', train=False)")



Processing TEST split
Label file: /content/drive/MyDrive/MAVERIC/maveric_cache/voc2007/datasets/elevater/voc2007_temp/voc2007_20211007/test_ic.txt
Split directory: /content/drive/MyDrive/MAVERIC/maveric_cache/voc2007/datasets/elevater/voc2007_temp/voc2007_20211007/test
  Found JPEGImages: /content/drive/MyDrive/MAVERIC/maveric_cache/voc2007/datasets/elevater/voc2007_temp/voc2007_20211007/test/VOCdevkit 2/VOC2007/JPEGImages
Found 4952 image-label pairs

Copying images to class folders...
  Progress: 500/4952 images
  Progress: 1000/4952 images
  Progress: 1500/4952 images
  Progress: 2000/4952 images
  Progress: 2500/4952 images
  Progress: 3000/4952 images
  Progress: 3500/4952 images
  Progress: 4000/4952 images
  Progress: 4500/4952 images

✓ Copied 4952 images

Processing TRAIN split
Label file: /content/drive/MyDrive/MAVERIC/maveric_cache/voc2007/datasets/elevater/voc2007_temp/voc2007_20211007/train_ic.txt
Split directory: /content/drive/MyDrive/MAVERIC/maveric_cache/voc2007/datas

In [ ]:
#!/usr/bin/env python3
"""
Download and organize Hateful Memes dataset from HuggingFace for MAVERIC.

Dataset: ytaek-oh/elevater/hateful_memes_20211014
Files:
  - img.zip (4.23 GB) - All images
  - test_meta.json (161 kB) - Test metadata
  - train_meta.json (2.83 MB) - Train metadata

JSON Format:
  {
    "images": [{"id": 1, "file_name": "img.zip@08291.png", ...}],
    "annotations": [{"id": 1, "category_id": 2, "image_id": 1, "text": "..."}]
  }

Usage: python download_hateful_memes.py [--output_dir ./data]
"""

import os
import argparse
import zipfile
import json
from pathlib import Path
from huggingface_hub import hf_hub_download
import shutil


# Hateful Memes classes (2 classes)
HATEFUL_MEMES_CLASSES = [
    'meme',              # category_id: 1
    'hatespeech meme'    # category_id: 2
]


def download_file(repo_id, subfolder, filename, output_dir):
    """Download a single file from HuggingFace."""
    print(f"  Downloading: {filename}")

    file_path = f"{subfolder}/{filename}"

    local_path = hf_hub_download(
        repo_id=repo_id,
        repo_type="dataset",
        filename=file_path,
        local_dir=output_dir,
        local_dir_use_symlinks=False
    )

    return Path(local_path)


def download_hateful_memes_files(repo_id="ytaek-oh/elevater",
                                  subfolder="hateful_memes_20211014",
                                  output_dir="./data"):
    """Download Hateful Memes files from HuggingFace."""
    print("="*70)
    print("Downloading Hateful Memes from HuggingFace")
    print("="*70)
    print(f"Repository: {repo_id}")
    print(f"Subfolder: {subfolder}")
    print()

    output_path = Path(output_dir) / "hateful_memes_temp"
    output_path.mkdir(parents=True, exist_ok=True)

    # Files to download
    files_to_download = [
        'img.zip',
        'test_meta.json',
        'train_meta.json'
    ]

    downloaded_files = {}

    for filename in files_to_download:
        try:
            local_path = download_file(repo_id, subfolder, filename, output_path)
            downloaded_files[filename] = local_path
            print(f"    ✓ Downloaded: {filename}")
        except Exception as e:
            print(f"    ⚠️  Failed to download {filename}: {e}")

    print(f"\n✅ Downloaded {len(downloaded_files)} files")
    return output_path / subfolder, downloaded_files


def extract_zip_file(zip_file):
    """Extract the img.zip file."""
    print("\n" + "="*70)
    print("Extracting ZIP file")
    print("="*70)

    print(f"\nExtracting: {zip_file.name}")

    extract_dir = zip_file.parent / "images"
    extract_dir.mkdir(exist_ok=True)

    try:
        with zipfile.ZipFile(zip_file, 'r') as zip_ref:
            zip_ref.extractall(extract_dir)

        # Count extracted files
        image_count = len(list(extract_dir.glob('*.png'))) + len(list(extract_dir.glob('*.jpg')))
        print(f"  ✓ Extracted {image_count} images to: {extract_dir.name}/")

        return extract_dir

    except Exception as e:
        print(f"  ❌ Error extracting: {e}")
        return None


def read_metadata(json_file):
    """
    Read metadata from JSON file.

    Returns: dict mapping image_id to (filename, category_id)
    """
    with open(json_file, 'r') as f:
        data = json.load(f)

    # Create image_id to filename mapping
    image_map = {}
    for img in data.get('images', []):
        img_id = img['id']
        filename = img['file_name']

        # Remove "img.zip@" prefix
        if '@' in filename:
            filename = filename.split('@', 1)[1]

        image_map[img_id] = filename

    # Create image_id to category_id mapping
    annotations = {}
    for ann in data.get('annotations', []):
        img_id = ann['image_id']
        category_id = ann['category_id']

        if img_id in image_map:
            filename = image_map[img_id]
            annotations[filename] = category_id

    return annotations


def organize_hateful_memes(downloaded_files, images_dir, output_dir="./data"):
    """Organize Hateful Memes dataset into class folders."""
    print("\n" + "="*70)
    print("Organizing Hateful Memes dataset for MAVERIC")
    print("="*70)

    target_dir = Path(output_dir) / "hateful_memes"

    # Create class directories
    print("\n📁 Creating directory structure...")
    for split in ['test', 'train']:
        for class_name in HATEFUL_MEMES_CLASSES:
            (target_dir / split / class_name).mkdir(parents=True, exist_ok=True)

    print(f"  ✓ Created {len(HATEFUL_MEMES_CLASSES)} class directories for each split")

    stats = {'test': 0, 'train': 0}

    # Process each split
    for split in ['test', 'train']:
        json_file = downloaded_files.get(f'{split}_meta.json')

        if not json_file:
            print(f"\n⚠️  Skipping {split} split - metadata file not found")
            continue

        print(f"\n📋 Processing {split} split...")
        print(f"  Metadata: {json_file.name}")

        # Read metadata
        annotations = read_metadata(json_file)
        print(f"  Found {len(annotations)} annotated images")

        # Copy images to class folders
        copied = 0
        skipped = 0

        for filename, category_id in annotations.items():
            # Convert category_id to class name (1-indexed to 0-indexed)
            class_idx = category_id - 1

            if class_idx < 0 or class_idx >= len(HATEFUL_MEMES_CLASSES):
                skipped += 1
                continue

            class_name = HATEFUL_MEMES_CLASSES[class_idx]

            # Find source image
            source_img = images_dir / filename

            if not source_img.exists():
                skipped += 1
                continue

            # Copy to target class directory
            target_img = target_dir / split / class_name / filename

            try:
                shutil.copy2(source_img, target_img)
                copied += 1

                if copied % 500 == 0:
                    print(f"    Progress: {copied}/{len(annotations)} images")

            except Exception as e:
                skipped += 1

        print(f"  ✓ Copied {copied} images")
        if skipped > 0:
            print(f"  ⚠️  Skipped {skipped} images")

        stats[split] = copied

    # Print summary
    print_summary(target_dir, stats)

    return target_dir


def print_summary(target_dir, stats):
    """Print dataset summary."""
    print("\n" + "="*70)
    print("Dataset Summary")
    print("="*70)

    for split in ['test', 'train']:
        split_dir = target_dir / split
        print(f"\n{split.capitalize()} Split:")

        total = 0
        for class_name in HATEFUL_MEMES_CLASSES:
            class_dir = split_dir / class_name
            if class_dir.exists():
                n_images = len(list(class_dir.glob('*.png')) +
                             list(class_dir.glob('*.jpg')))

                if n_images > 0:
                    print(f"  {class_name:20s}: {n_images:4d} images")
                    total += n_images

        print(f"  {'Total':20s}: {total:4d} images")


def cleanup_temp_files(temp_dir):
    """Clean up temporary download directory."""
    print("\n🧹 Cleaning up temporary files...")

    try:
        if temp_dir.exists():
            shutil.rmtree(temp_dir)
            print(f"  ✓ Removed: {temp_dir}")
    except Exception as e:
        print(f"  ⚠️  Could not remove temp dir: {e}")

print("="*70)
print("Hateful Memes Dataset Downloader for MAVERIC")
print("="*70)

# Download files
temp_path, downloaded_files = download_hateful_memes_files(output_dir="/content/drive/MyDrive/MAVERIC/maveric_cache/hatefulmemes/datasets/elevater/hatefulmemes/")

# Extract ZIP
zip_file = downloaded_files.get('img.zip')

images_dir = extract_zip_file(zip_file)

# Organize dataset
target_dir = organize_hateful_memes(downloaded_files, images_dir, "/content/drive/MyDrive/MAVERIC/maveric_cache/hatefulmemes/datasets/elevater/hatefulmemes/")

# Final message
print("\n" + "="*70)
print("✅ Hateful Memes dataset ready for MAVERIC!")
print("="*70)

Hateful Memes Dataset Downloader for MAVERIC
Repository: ytaek-oh/elevater
Subfolder: hateful_memes_20211014

  Downloading: img.zip


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:202: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(


hateful_memes_20211014/img.zip:   0%|          | 0.00/4.23G [00:00<?, ?B/s]

    ✓ Downloaded: img.zip
  Downloading: test_meta.json


test_meta.json: 0.00B [00:00, ?B/s]

    ✓ Downloaded: test_meta.json
  Downloading: train_meta.json


train_meta.json: 0.00B [00:00, ?B/s]

    ✓ Downloaded: train_meta.json

✅ Downloaded 3 files

Extracting ZIP file

Extracting: img.zip
  ✓ Extracted 12140 images to: images/

Organizing Hateful Memes dataset for MAVERIC

📁 Creating directory structure...
  ✓ Created 2 class directories for each split

📋 Processing test split...
  Metadata: test_meta.json
  Found 500 annotated images
    Progress: 500/500 images
  ✓ Copied 500 images

📋 Processing train split...
  Metadata: train_meta.json
  Found 8500 annotated images
    Progress: 500/8500 images
    Progress: 1000/8500 images
    Progress: 1500/8500 images
    Progress: 2000/8500 images
    Progress: 2500/8500 images
    Progress: 3000/8500 images
    Progress: 3500/8500 images
    Progress: 4000/8500 images
    Progress: 4500/8500 images
    Progress: 5000/8500 images
    Progress: 5500/8500 images
    Progress: 6000/8500 images
    Progress: 6500/8500 images
    Progress: 7000/8500 images
    Progress: 7500/8500 images
    Progress: 8000/8500 images
    Progress: 8500

In [ ]:
#!/usr/bin/env python3
"""
Download and organize RESISC45 dataset from HuggingFace for MAVERIC.

Dataset: ytaek-oh/elevater/resisc45_clip_20210924
Files:
  - images.zip (431 MB) - All images
  - test.json (3.85 MB) - Test metadata
  - train.json (890 kB) - Train metadata
  - val.json (475 kB) - Validation metadata

JSON Format:
  {
    "categories": [{"id": 1, "name": "airplane"}, ...],
    "images": [{"id": 1, "file_name": "images.zip@freeway/freeway_450.jpg", ...}],
    "annotations": [{"id": 1, "category_id": 15, "image_id": 1}]
  }

Usage: python download_resisc45.py [--output_dir ./data]
"""

import os
import argparse
import zipfile
import json
from pathlib import Path
from huggingface_hub import hf_hub_download
import shutil


# RESISC45 class names (45 classes) - from elevater_datasets.py
RESISC45_CLASSES = [
    'airplane',
    'airport',
    'baseball diamond',
    'basketball court',
    'beach',
    'bridge',
    'chaparral',
    'church',
    'circular farmland',
    'cloud',
    'commercial area',
    'dense residential',
    'desert',
    'forest',
    'freeway',
    'golf course',
    'ground track field',
    'harbor',
    'industrial area',
    'intersection',
    'island',
    'lake',
    'meadow',
    'medium residential',
    'mobile home park',
    'mountain',
    'overpass',
    'palace',
    'parking lot',
    'railway',
    'railway station',
    'rectangular farmland',
    'river',
    'roundabout',
    'runway',
    'sea ice',
    'ship',
    'snowberg',
    'sparse residential',
    'stadium',
    'storage tank',
    'tennis court',
    'terrace',
    'thermal power station',
    'wetland'
]


def download_file(repo_id, subfolder, filename, output_dir):
    """Download a single file from HuggingFace."""
    print(f"  Downloading: {filename}")

    file_path = f"{subfolder}/{filename}"

    local_path = hf_hub_download(
        repo_id=repo_id,
        repo_type="dataset",
        filename=file_path,
        local_dir=output_dir,
        local_dir_use_symlinks=False
    )

    return Path(local_path)


def download_resisc45_files(repo_id="ytaek-oh/elevater",
                            subfolder="resisc45_clip_20210924",
                            output_dir="./data"):
    """Download RESISC45 files from HuggingFace."""
    print("="*70)
    print("Downloading RESISC45 from HuggingFace")
    print("="*70)
    print(f"Repository: {repo_id}")
    print(f"Subfolder: {subfolder}")
    print()

    output_path = Path(output_dir) / "resisc45_temp"
    output_path.mkdir(parents=True, exist_ok=True)

    # Files to download
    files_to_download = [
        'images.zip',
        'test.json',
        'train.json',
        'val.json'
    ]

    downloaded_files = {}

    for filename in files_to_download:
        try:
            local_path = download_file(repo_id, subfolder, filename, output_path)
            downloaded_files[filename] = local_path
            print(f"    ✓ Downloaded: {filename}")
        except Exception as e:
            print(f"    ⚠️  Failed to download {filename}: {e}")

    print(f"\n✅ Downloaded {len(downloaded_files)} files")
    return output_path / subfolder, downloaded_files


def extract_zip_file(zip_file):
    """Extract the images.zip file."""
    print("\n" + "="*70)
    print("Extracting ZIP file")
    print("="*70)

    print(f"\nExtracting: {zip_file.name}")

    extract_dir = zip_file.parent / "images"
    extract_dir.mkdir(exist_ok=True)

    try:
        # with zipfile.ZipFile(zip_file, 'r') as zip_ref:
        #     zip_ref.extractall(extract_dir)

        # Count extracted files
        image_count = sum(1 for _ in extract_dir.rglob('*.jpg'))
        print(f"  ✓ Extracted {image_count} images to: {extract_dir.name}/")

        return extract_dir

    except Exception as e:
        print(f"  ❌ Error extracting: {e}")
        return None


def read_metadata(json_file):
    """
    Read metadata from JSON file.

    Returns: tuple of (category_map, image_annotations)
      - category_map: dict mapping category_id to category name
      - image_annotations: dict mapping image_id to (filename, category_id)
    """
    with open(json_file, 'r') as f:
        data = json.load(f)

    # Create category_id to name mapping
    category_map = {}
    for cat in data.get('categories', []):
        cat_id = cat['id']
        cat_name = cat['name']
        # Replace underscore with space to match MAVERIC class names
        cat_name = cat_name.replace('_', ' ')
        category_map[cat_id] = cat_name

    # Create image_id to filename mapping
    image_map = {}
    for img in data.get('images', []):
        img_id = img['id']
        filename = img['file_name']

        # Remove "images.zip@" prefix
        if '@' in filename:
            filename = filename.split('@', 1)[1]

        image_map[img_id] = filename

    # Create image_id to (filename, category_id) mapping
    annotations = {}
    for ann in data.get('annotations', []):
        img_id = ann['image_id']
        category_id = ann['category_id']

        if img_id in image_map:
            filename = image_map[img_id]
            annotations[img_id] = (filename, category_id)

    return category_map, annotations


def organize_resisc45(downloaded_files, images_dir, output_dir="./data"):
    """Organize RESISC45 dataset into class folders."""
    print("\n" + "="*70)
    print("Organizing RESISC45 dataset for MAVERIC")
    print("="*70)

    target_dir = Path(output_dir) / "resisc45"

    # Create class directories
    print("\n📁 Creating directory structure...")
    for split in ['test', 'train', 'val']:
        for class_name in RESISC45_CLASSES:
            (target_dir / split / class_name).mkdir(parents=True, exist_ok=True)

    print(f"  ✓ Created {len(RESISC45_CLASSES)} class directories for each split")

    stats = {'test': 0, 'train': 0, 'val': 0}

    # Process each split
    for split in ['test', 'train', 'val']:
        json_file = downloaded_files.get(f'{split}.json')

        if not json_file:
            print(f"\n⚠️  Skipping {split} split - metadata file not found")
            continue

        print(f"\n📋 Processing {split} split...")
        print(f"  Metadata: {json_file.name}")

        # Read metadata
        category_map, annotations = read_metadata(json_file)
        print(f"  Found {len(annotations)} annotated images")
        print(f"  Categories: {len(category_map)}")

        # Copy images to class folders
        copied = 0
        skipped = 0

        for img_id, (filename, category_id) in annotations.items():
            # Get category name
            category_name = category_map.get(category_id)

            if not category_name or category_name not in RESISC45_CLASSES:
                skipped += 1
                continue

            # Find source image - filename already contains path like "freeway/freeway_450.jpg"
            source_img = images_dir / filename

            if not source_img.exists():
                # Try alternative: look in all subdirectories
                img_basename = Path(filename).name
                found_files = list(images_dir.rglob(img_basename))
                if found_files:
                    source_img = found_files[0]
                else:
                    skipped += 1
                    continue

            # Copy to target class directory
            target_img = target_dir / split / category_name / source_img.name

            try:
                shutil.copy2(source_img, target_img)
                copied += 1

                if copied % 1000 == 0:
                    print(f"    Progress: {copied}/{len(annotations)} images")

            except Exception as e:
                skipped += 1

        print(f"  ✓ Copied {copied} images")
        if skipped > 0:
            print(f"  ⚠️  Skipped {skipped} images")

        stats[split] = copied

    # Print summary
    print_summary(target_dir, stats)

    return target_dir


def print_summary(target_dir, stats):
    """Print dataset summary."""
    print("\n" + "="*70)
    print("Dataset Summary")
    print("="*70)

    for split in ['test', 'train', 'val']:
        split_dir = target_dir / split

        if not split_dir.exists():
            continue

        print(f"\n{split.capitalize()} Split:")

        total = 0
        class_counts = []

        for class_name in RESISC45_CLASSES:
            class_dir = split_dir / class_name
            if class_dir.exists():
                n_images = len(list(class_dir.glob('*.jpg')))

                if n_images > 0:
                    class_counts.append((class_name, n_images))
                    total += n_images

        # Show top 10 classes
        class_counts.sort(key=lambda x: x[1], reverse=True)
        for class_name, count in class_counts[:10]:
            print(f"  {class_name:25s}: {count:4d} images")

        if len(class_counts) > 10:
            print(f"  ... and {len(class_counts) - 10} more classes")

        print(f"  {'Total':25s}: {total:4d} images")
        print(f"  {'Classes with data':25s}: {len(class_counts)}/{len(RESISC45_CLASSES)}")


def cleanup_temp_files(temp_dir):
    """Clean up temporary download directory."""
    print("\n🧹 Cleaning up temporary files...")

    try:
        if temp_dir.exists():
            shutil.rmtree(temp_dir)
            print(f"  ✓ Removed: {temp_dir}")
    except Exception as e:
        print(f"  ⚠️  Could not remove temp dir: {e}")


print("="*70)
print("RESISC45 Dataset Downloader for MAVERIC")
print("="*70)

# Download files
temp_path, downloaded_files = download_resisc45_files(output_dir="/content/drive/MyDrive/MAVERIC/maveric_cache/resisc45/datasets/elevater/resisc45/")

# Extract ZIP
zip_file = downloaded_files.get('images.zip')

images_dir = extract_zip_file(zip_file)

# Organize dataset
target_dir = organize_resisc45(downloaded_files, images_dir, "/content/drive/MyDrive/MAVERIC/maveric_cache/resisc45/datasets/elevater/resisc45/")

# Final message
print("\n" + "="*70)
print("✅ RESISC45 dataset ready for MAVERIC!")
print("="*70)
print(f"\n📂 Dataset location: {target_dir}")
print("\nNext steps:")
print("  1. Verify the dataset structure")
print("  2. Use with MAVERIC:")
print("     from maveric.datasets.elevater_datasets import ELEVATERDataset")
print("     dataset = ELEVATERDataset('resisc45', root='./data', train=False)")
print("  3. Run your evaluation experiments")


RESISC45 Dataset Downloader for MAVERIC
Repository: ytaek-oh/elevater
Subfolder: resisc45_clip_20210924

  Downloading: images.zip
    ✓ Downloaded: images.zip
  Downloading: test.json
    ✓ Downloaded: test.json
  Downloading: train.json
    ✓ Downloaded: train.json
  Downloading: val.json
    ✓ Downloaded: val.json

✅ Downloaded 4 files

Extracting ZIP file

Extracting: images.zip
  ✓ Extracted 31500 images to: images/

Organizing RESISC45 dataset for MAVERIC

📁 Creating directory structure...
  ✓ Created 45 class directories for each split

📋 Processing test split...
  Metadata: test.json
  Found 25200 annotated images
  Categories: 45
    Progress: 1000/25200 images
    Progress: 2000/25200 images
    Progress: 3000/25200 images
    Progress: 4000/25200 images
    Progress: 5000/25200 images
    Progress: 6000/25200 images
    Progress: 7000/25200 images
    Progress: 8000/25200 images
    Progress: 9000/25200 images
    Progress: 10000/25200 images
    Progress: 11000/25200 images